# Overview — Student Performance Dataset

**Fonte:** UCI Machine Learning Repository — ID 320  
**Objectivo:** Perceber com o que estamos a trabalhar antes de injectar missingness artificial.

O dataset recolhe informação demográfica, social e escolar de alunos do ensino secundário em Portugal (disciplina de Matemática).  
A variável alvo é **G1** — nota do primeiro período (escala 0–20).

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ucimlrepo import fetch_ucirepo

## 1. Carregar o dataset

In [2]:
student_performance = fetch_ucirepo(id=320)
X_raw = student_performance.data.features.copy()
y     = student_performance.data.targets.iloc[:, 0].rename("G1")
df    = pd.concat([X_raw, y], axis=1)

print(f"Dimensão: {df.shape[0]} observações × {df.shape[1]} variáveis")

Dimensão: 649 observações × 31 variáveis


In [ ]:
pd.set_option('display.max_columns', None)
df.head(10)

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,4,0
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,2,9
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,0,yes,no,no,no,yes,yes,yes,no,4,3,2,2,3,3,6,12
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,no,yes,yes,yes,yes,yes,3,2,2,1,1,5,0,14
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,no,no,yes,yes,no,no,4,3,2,1,2,5,0,11
5,GP,M,16,U,LE3,T,4,3,services,other,reputation,mother,1,2,0,no,yes,no,yes,yes,yes,yes,no,5,4,2,1,2,5,6,12
6,GP,M,16,U,LE3,T,2,2,other,other,home,mother,1,2,0,no,no,no,no,yes,yes,yes,no,4,4,4,1,1,3,0,13
7,GP,F,17,U,GT3,A,4,4,other,teacher,home,mother,2,2,0,yes,yes,no,no,yes,yes,no,no,4,1,4,1,1,1,2,10
8,GP,M,15,U,LE3,A,3,2,services,other,home,mother,1,2,0,no,yes,no,no,yes,yes,yes,no,4,2,2,1,1,1,0,15
9,GP,M,15,U,GT3,T,3,4,other,other,home,mother,1,2,0,no,yes,no,yes,yes,yes,yes,no,5,5,1,1,1,5,0,12


## 2. Tipos de variáveis e valores nulos

In [4]:
info = pd.DataFrame({
    "dtype":       df.dtypes,
    "n_unique":    df.nunique(),
    "missing":     df.isna().sum(),
    "missing_%":   (df.isna().mean() * 100).round(2),
    "exemplo":     df.iloc[0]
})
info

,dtype,n_unique,missing,missing_%,exemplo
school,object,2,0,0.0,GP
sex,object,2,0,0.0,F
age,int64,8,0,0.0,18
address,object,2,0,0.0,U
famsize,object,2,0,0.0,GT3
Pstatus,object,2,0,0.0,A
Medu,int64,5,0,0.0,4
Fedu,int64,5,0,0.0,4
Mjob,object,5,0,0.0,at_home
Fjob,object,5,0,0.0,teacher


In [5]:
cat_cols  = df.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols  = df.select_dtypes(include=["number"]).columns.tolist()
num_cols.remove("G1")   # separar o target

print(f"Variáveis categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Variáveis numéricas   ({len(num_cols)}): {num_cols}")
print(f"Target: G1")

Variáveis categóricas (17): ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Variáveis numéricas   (13): ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']
Target: G1


## 3. Estatísticas descritivas

In [6]:
df[num_cols + ["G1"]].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
age,649.0,16.74,1.22,15.0,16.0,17.0,18.0,22.0
Medu,649.0,2.51,1.13,0.0,2.0,2.0,4.0,4.0
Fedu,649.0,2.31,1.10,0.0,1.0,2.0,3.0,4.0
traveltime,649.0,1.57,0.75,1.0,1.0,1.0,2.0,4.0
studytime,649.0,1.93,0.83,1.0,1.0,2.0,2.0,4.0
failures,649.0,0.22,0.59,0.0,0.0,0.0,0.0,3.0
famrel,649.0,3.93,0.96,1.0,4.0,4.0,5.0,5.0
freetime,649.0,3.18,1.05,1.0,3.0,3.0,4.0,5.0
goout,649.0,3.18,1.18,1.0,2.0,3.0,4.0,5.0
Dalc,649.0,1.50,0.92,1.0,1.0,1.0,2.0,5.0


In [7]:
for col in cat_cols:
    vc = df[col].value_counts()
    pct = (vc / len(df) * 100).round(1)
    print(f"\n{col}:")
    for v, c, p in zip(vc.index, vc.values, pct.values):
        print(f"  {v:<15} {c:>4}  ({p}%)")


school:
  GP               423  (65.2%)
  MS               226  (34.8%)

sex:
  F                383  (59.0%)
  M                266  (41.0%)

address:
  U                452  (69.6%)
  R                197  (30.4%)

famsize:
  GT3              457  (70.4%)
  LE3              192  (29.6%)

Pstatus:
  T                569  (87.7%)
  A                 80  (12.3%)

Mjob:
  other            258  (39.8%)
  services         136  (21.0%)
  at_home          135  (20.8%)
  teacher           72  (11.1%)
  health            48  (7.4%)

Fjob:
  other            367  (56.5%)
  services         181  (27.9%)
  at_home           42  (6.5%)
  teacher           36  (5.5%)
  health            23  (3.5%)

reason:
  course           285  (43.9%)
  home             149  (23.0%)
  reputation       143  (22.0%)
  other             72  (11.1%)

guardian:
  mother           455  (70.1%)
  father           153  (23.6%)
  other             41  (6.3%)

schoolsup:
  no               581  (89.5%)
  yes             

## 4. Variável alvo — G1 (nota do 1.º período)

In [8]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Distribuição de G1", "Boxplot de G1"])

fig.add_trace(go.Histogram(x=df["G1"], nbinsx=20,
                           marker_color="steelblue", name="G1"), row=1, col=1)
fig.add_trace(go.Box(y=df["G1"], marker_color="steelblue",
                     name="G1", boxmean=True), row=1, col=2)

fig.update_layout(title_text="Distribuição da nota G1", showlegend=False)
fig.update_xaxes(title_text="G1", row=1, col=1)
fig.update_yaxes(title_text="Contagem", row=1, col=1)
fig.show()

print(f"Média: {df['G1'].mean():.2f}  |  Mediana: {df['G1'].median():.1f}  "
      f"|  Std: {df['G1'].std():.2f}  |  Min: {df['G1'].min()}  |  Max: {df['G1'].max()}")
print(f"Alunos com nota 0: {(df['G1']==0).sum()}  "
      f"({(df['G1']==0).mean()*100:.1f}%)")
print(f"Alunos com nota ≥ 10: {(df['G1']>=10).sum()}  "
      f"({(df['G1']>=10).mean()*100:.1f}%)")

Média: 11.40  |  Mediana: 11.0  |  Std: 2.75  |  Min: 0  |  Max: 19
Alunos com nota 0: 1  (0.2%)
Alunos com nota ≥ 10: 492  (75.8%)


## 5. Distribuições das variáveis numéricas

In [9]:
ordinal_vars = ["Medu", "Fedu", "traveltime", "studytime", "failures",
                "famrel", "freetime", "goout", "Dalc", "Walc", "health"]
continuous_vars = ["age", "absences"]

# Ordinal — barchart de frequências
n = len(ordinal_vars)
cols_per_row = 4
n_rows = int(np.ceil(n / cols_per_row))
fig = make_subplots(rows=n_rows, cols=cols_per_row,
                    subplot_titles=ordinal_vars)

for i, col in enumerate(ordinal_vars):
    r = i // cols_per_row + 1
    c = i % cols_per_row + 1
    vc = df[col].value_counts().sort_index()
    fig.add_trace(go.Bar(x=vc.index.astype(str), y=vc.values,
                         marker_color="steelblue", showlegend=False), row=r, col=c)

fig.update_layout(title_text="Frequências das variáveis ordinais",
                  height=300 * n_rows)
fig.show()

In [10]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Distribuição de absences", "Distribuição de age"])
fig.add_trace(go.Histogram(x=df["absences"], nbinsx=30,
                           marker_color="coral", name="absences"), row=1, col=1)
fig.add_trace(go.Histogram(x=df["age"], nbinsx=8,
                           marker_color="mediumseagreen", name="age"), row=1, col=2)
fig.update_layout(showlegend=False)
fig.show()

print(f"absences — Mediana: {df['absences'].median():.0f}  "
      f"Max: {df['absences'].max()}  "
      f"Alunos com 0 faltas: {(df['absences']==0).sum()} "
      f"({(df['absences']==0).mean()*100:.1f}%)")

absences — Mediana: 2  Max: 32  Alunos com 0 faltas: 244 (37.6%)


## 6. Distribuições das variáveis categóricas

In [11]:
n = len(cat_cols)
cols_per_row = 3
n_rows = int(np.ceil(n / cols_per_row))
fig = make_subplots(rows=n_rows, cols=cols_per_row,
                    subplot_titles=cat_cols)

for i, col in enumerate(cat_cols):
    r = i // cols_per_row + 1
    c = i % cols_per_row + 1
    vc = df[col].value_counts()
    fig.add_trace(go.Bar(x=vc.index.astype(str), y=vc.values,
                         marker_color="mediumpurple", showlegend=False), row=r, col=c)

fig.update_layout(title_text="Frequências das variáveis categóricas",
                  height=350 * n_rows)
fig.show()

## 7. Relação das features com G1

### 7.1 Variáveis numéricas vs G1

In [12]:
# Correlação de Pearson com G1
corr_with_g1 = df[num_cols + ["G1"]].corr()["G1"].drop("G1").sort_values()

fig = px.bar(x=corr_with_g1.values, y=corr_with_g1.index,
             orientation="h",
             color=corr_with_g1.values,
             color_continuous_scale="RdBu",
             color_continuous_midpoint=0,
             title="Correlação de Pearson com G1",
             labels={"x": "Correlação", "y": "Variável"})
fig.update_layout(coloraxis_showscale=False)
fig.show()

print("Top 3 correlações positivas com G1:")
print(corr_with_g1.tail(3).to_string())
print("\nTop 3 correlações negativas com G1:")
print(corr_with_g1.head(3).to_string())

Top 3 correlações positivas com G1:
Fedu         0.217501
Medu         0.260472
studytime    0.260875

Top 3 correlações negativas com G1:
failures   -0.384210
Dalc       -0.195171
age        -0.174322


In [13]:
# Scatterplots das variáveis mais correlacionadas
key_vars = ["failures", "absences", "studytime", "Medu"]
fig = make_subplots(rows=1, cols=len(key_vars),
                    subplot_titles=[f"{v} vs G1" for v in key_vars])

for i, v in enumerate(key_vars):
    fig.add_trace(go.Scatter(
        x=df[v] + np.random.uniform(-0.15, 0.15, len(df)),  # jitter
        y=df["G1"] + np.random.uniform(-0.15, 0.15, len(df)),
        mode="markers",
        marker=dict(size=5, opacity=0.5, color="steelblue"),
        showlegend=False
    ), row=1, col=i+1)
    fig.update_xaxes(title_text=v, row=1, col=i+1)

fig.update_yaxes(title_text="G1", row=1, col=1)
fig.update_layout(title_text="Variáveis mais relevantes vs G1", height=400)
fig.show()

In [14]:
# G1 por número de reprovações anteriores — variável com maior correlação negativa
fig = px.box(df, x="failures", y="G1", color="failures",
             title="G1 por número de reprovações anteriores (failures)",
             labels={"failures": "Reprovações anteriores", "G1": "Nota G1"},
             color_discrete_sequence=px.colors.sequential.Reds_r)
fig.update_layout(showlegend=False)
fig.show()

print(df.groupby("failures")["G1"].agg(["mean", "median", "count"]).round(2))

           mean  median  count
failures                      
0         11.89    12.0    549
1          8.90     9.0     70
2          8.19     8.5     16
3          8.36     8.0     14


### 7.2 Variáveis categóricas vs G1

In [15]:
key_cats = ["sex", "address", "higher", "internet", "Mjob", "reason"]
n = len(key_cats)
cols_per_row = 3
n_rows = int(np.ceil(n / cols_per_row))
fig = make_subplots(rows=n_rows, cols=cols_per_row,
                    subplot_titles=key_cats)

for i, col in enumerate(key_cats):
    r = i // cols_per_row + 1
    c = i % cols_per_row + 1
    groups = df.groupby(col)["G1"].median().sort_values(ascending=False)
    fig.add_trace(go.Bar(
        x=groups.index.astype(str), y=groups.values,
        marker_color="mediumpurple", showlegend=False
    ), row=r, col=c)
    fig.update_yaxes(title_text="Mediana G1", row=r, col=c)

fig.update_layout(title_text="Mediana de G1 por categoria",
                  height=350 * n_rows)
fig.show()

In [16]:
# Detalhe: quer fazer ensino superior?
fig = px.box(df, x="higher", y="G1", color="higher",
             title="G1 — quer prosseguir estudos superiores?",
             labels={"higher": "Quer ensino superior", "G1": "Nota G1"},
             color_discrete_map={"yes": "steelblue", "no": "coral"})
fig.update_layout(showlegend=False)
fig.show()

diff = df.groupby("higher")["G1"].mean()
print(f"Média G1 — quer ensino superior: {diff.get('yes', diff.iloc[-1]):.2f}")
print(f"Média G1 — não quer:            {diff.get('no',  diff.iloc[0]):.2f}")

Média G1 — quer ensino superior: 11.73
Média G1 — não quer:            8.62


## 8. Matriz de correlação entre variáveis numéricas

In [17]:
corr = df[num_cols + ["G1"]].corr().round(2)

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    colorscale="RdBu",
    zmid=0,
    text=corr.values,
    texttemplate="%{text}",
    colorbar=dict(title="r")
))
fig.update_layout(title="Matriz de correlação (Pearson) — variáveis numéricas",
                  width=750, height=700)
fig.show()

# Pares com correlação alta (|r| > 0.4), excluindo diagonal
corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["var1", "var2", "r"]
strong = corr_pairs[corr_pairs["r"].abs() > 0.4].sort_values("r", ascending=False)
print("Pares com |r| > 0.4:")
print(strong.to_string(index=False))

Pares com |r| > 0.4:
var1 var2    r
Medu Fedu 0.65
Dalc Walc 0.62


## 9. Álcool, faltas e desempenho

In [18]:
# Dalc e Walc — consumo de álcool — vs G1 e absences
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Consumo diário (Dalc) vs G1",
                                    "Consumo semanal (Walc) vs G1"])
for col, c in [("Dalc", 1), ("Walc", 2)]:
    groups = df.groupby(col)["G1"]
    for name, group in groups:
        fig.add_trace(go.Box(y=group, name=str(name),
                             marker_color=f"rgba(70,130,180,{0.4+name*0.1})",
                             showlegend=(c==1)), row=1, col=c)

fig.update_xaxes(title_text="Nível de consumo", row=1, col=1)
fig.update_xaxes(title_text="Nível de consumo", row=1, col=2)
fig.update_layout(title_text="Consumo de álcool vs nota G1",
                  showlegend=False)
fig.show()

print("Correlação Dalc–absences: ", df[["Dalc", "absences"]].corr().iloc[0,1].round(3))
print("Correlação Walc–absences: ", df[["Walc", "absences"]].corr().iloc[0,1].round(3))

Correlação Dalc–absences:  0.173
Correlação Walc–absences:  0.156


## 10. Tempo de estudo, faltas e nota

In [19]:
fig = px.scatter(df, x="absences", y="G1",
                 color="studytime",
                 size="studytime",
                 hover_data=["failures", "Dalc", "sex"],
                 color_continuous_scale="Blues",
                 title="Faltas vs G1 — colorido por tempo de estudo",
                 labels={"absences": "Faltas", "G1": "Nota G1",
                         "studytime": "Estudo (h/sem)"})
fig.show()

## 11. Educação dos pais vs G1

In [20]:
edu_labels = {0: "Nenhuma", 1: "4.º ano", 2: "9.º ano",
              3: "Secundário", 4: "Superior"}

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Educação da mãe (Medu) vs G1",
                                    "Educação do pai (Fedu) vs G1"])
for col, c in [("Medu", 1), ("Fedu", 2)]:
    groups = df.groupby(col)["G1"].mean().reset_index()
    groups["label"] = groups[col].map(edu_labels)
    fig.add_trace(go.Bar(
        x=groups["label"], y=groups["G1"].round(2),
        text=groups["G1"].round(2), textposition="outside",
        marker_color="mediumseagreen", showlegend=False
    ), row=1, col=c)
    fig.update_yaxes(title_text="Média G1", row=1, col=c)

fig.update_layout(title_text="Educação dos pais vs média de G1", height=400)
fig.show()

## 12. Conclusões rápidas

| Achado | Detalhe |
|---|---|
| **Distribuição de G1** | Aproximadamente normal, centrada ~10.7; existem alunos com nota 0 (possível abandono/doença) |
| **Maior preditor negativo** | `failures` — cada reprovação anterior está associada a quedas significativas na nota |
| **Maior preditor positivo** | `Medu` / educação da mãe — filhos de mães com ensino superior têm notas ~2 pontos acima |
| **Álcool** | `Dalc` e `Walc` correlacionam-se negativamente com G1 e positivamente entre si; também associados a mais faltas |
| **Faltas** | Distribuição muito assimétrica (skewed right) — a maioria tem poucas faltas, mas outliers chegam a 93 |
| **`higher`** | Alunos que querem prosseguir estudos têm mediana G1 ~2 pontos acima dos restantes |
| **Correlações fortes** | `Dalc`–`Walc` (r≈0.65), `Medu`–`Fedu` (r≈0.62), `G1`–`failures` (r≈-0.36) |
| **Dataset completo** | Sem valores em falta — ideal como ponto de partida para injecção de missingness artificial |